# 06D - XGBoost Classifier (Execution-Ready)
Optimized notebook for GitHub. Run **Run All** to generate and save outputs.

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import *

df=pd.read_csv("american_bankruptcy.csv")
df["target"]=df["status_label"].map({"alive":0,"failed":1})
display(df.head())
print(df.shape)


## Data Preparation

In [ ]:
drop=["status_label","target"]
if "company_name" in df.columns:
    drop.append("company_name")
X=df.drop(columns=drop)
y=df["target"]
num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42)

pre=ColumnTransformer([
("num",SimpleImputer(strategy="median"),num),
("cat",Pipeline([
("imp",SimpleImputer(strategy="most_frequent")),
("enc",OneHotEncoder(handle_unknown="ignore"))
]),cat)
])


## Train Model

In [ ]:
model=Pipeline([
("preprocessor",pre),
("classifier",XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
))
])

model.fit(X_train,y_train)


## Cross Validation

In [ ]:
cv=cross_validate(model,X_train,y_train,cv=3,
scoring=["accuracy","precision","recall","f1","roc_auc"])
display(pd.DataFrame(cv).describe())


## Evaluation

In [ ]:
pred=model.predict(X_test)
prob=model.predict_proba(X_test)[:,1]

results=pd.DataFrame({
"Metric":["Accuracy","Precision","Recall","F1","ROC-AUC"],
"Value":[
accuracy_score(y_test,pred),
precision_score(y_test,pred),
recall_score(y_test,pred),
f1_score(y_test,pred),
roc_auc_score(y_test,prob)
]})
display(results)
print(classification_report(y_test,pred))


## Visualizations

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()
RocCurveDisplay.from_predictions(y_test,prob)
plt.show()
PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.show()


## Feature Importance

In [ ]:
xgb=model.named_steps["classifier"]
features=list(num)
if len(cat):
    features.extend(model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["enc"].get_feature_names_out(cat))
imp=pd.DataFrame({"Feature":features,"Importance":xgb.feature_importances_}).sort_values("Importance",ascending=False)
display(imp.head(20))


## Save Model

In [ ]:
joblib.dump(model,"xgboost_model.joblib")
print("Saved xgboost_model.joblib")
